In [4]:
from read_model_runs import read_model_runs

# Ler dados completos
question_long_df, model_wide_accuracy_df, question_wide_accuracy_df, firac_order, model_order = read_model_runs('../data/processed/model-runs')

# Filtrar apenas as linhas em português
question_long_df = question_long_df[question_long_df["language"] == "portuguese"].copy()

print('shape', question_long_df.shape)
print('# unique questions', question_long_df['question_id'].nunique())
print('firac_order', firac_order)
print('model_order', model_order)

question_long_df = question_long_df[
    question_long_df["model_name"].str.contains("gemma", case=False, na=False) &
    ~question_long_df["model_name"].str.contains("gemma-3n", case=False, na=False)
]

question_long_df = question_long_df[question_long_df["firac"].ne("F____")]


question_long_df.head(1)


shape (44720, 30)
# unique questions 2496
firac_order ['FILA_', 'FIR__', 'FI___', 'FIL__', 'F____', '_____']
model_order ['gemini-2.5-flash-lite', 'gemini-2.0-flash-lite', 'gemma-3-27b-it', 'gemma-3n-e4b-it', 'gemma-3-12b-it', 'gemma-3n-e2b-it', 'gemma-3-4b-it']


,model_name,firac,language,is_correct,pdf_filename,question_id,materia,oab_test_id,oab_question_id,chosen_option,...,D,full_response,Facts,Issue,Rule,Application,Conclusion,rule_count,fact_count,response_time_seconds
1782,gemma-3-12b-it,FILA_,portuguese,True,oab-153.pdf,oab-153.pdf-007,DIREITO CIVIL,II,7,C,...,cada herdeiro pode ser demandado pela dívida t...,"{\n ""F"":[\n ""A existência de um regime de sol...",['A existência de um regime de solidariedade p...,Qual a correta aplicação das regras do regime ...,"Art. 276 do Código Civil, Art. 279 do Código C...",Ao analisar a responsabilidade em caso de fale...,NaN,4,5,10.862945


In [5]:
question_long_df.columns

Index(['model_name', 'firac', 'language', 'is_correct', 'pdf_filename',
       'question_id', 'materia', 'oab_test_id', 'oab_question_id',
       'chosen_option', 'correct_option', 'enunciado', 'prompt',
       'finish_reason', 'avg_logprobs', 'input_token_count',
       'output_token_count', 'A', 'B', 'C', 'D', 'full_response', 'Facts',
       'Issue', 'Rule', 'Application', 'Conclusion', 'rule_count',
       'fact_count', 'response_time_seconds'],
      dtype='object')

In [ ]:
import pandas as pd

# Agregação principal por questão
question_agg_df = (
    question_long_df
        .groupby("question_id", as_index=False)
        .agg(
            accuracy=("is_correct", "mean"),
            input_token_count=("input_token_count", "first"),
            output_token_count=("output_token_count", "mean"),
            fact_count=("fact_count", "first"),
            rule_count=("rule_count", "first"),
        )
)

'''
# Lê estatísticas de entropia
question_entropy_df = pd.read_csv("../data/processed/question_entropy.csv")

# Seleciona apenas colunas numéricas
entropy_numeric_cols = question_entropy_df.select_dtypes(include="number").columns

# Garante que question_id esteja presente
entropy_cols = ["question_id"] + list(entropy_numeric_cols)

question_entropy_df = question_entropy_df[entropy_cols]

# Merge
question_agg_df = question_agg_df.merge(
    question_entropy_df,
    on="question_id",
    how="left"
)
'''

question_agg_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/question_entropy.csv'

In [ ]:
counts = (
    question_long_df
        .groupby("question_id")
        .size()
        .reset_index(name="row_count")
        .sort_values("row_count", ascending=False)
)

# Total antes do filtro
total_question_ids = len(counts)

# Obtém o count máximo
max_count = counts["row_count"].max()

# Filtra apenas os question_id com o count máximo
question_ids = counts.loc[counts["row_count"] == max_count, "question_id"].tolist()

max_count, len(question_ids), total_question_ids, question_ids


In [ ]:
import matplotlib.pyplot as plt

# Gera o histograma da coluna 'row_count'
plt.figure(figsize=(10,6))
plt.hist(counts['row_count'], bins=20, color='skyblue', edgecolor='black')
plt.title('Histograma de row_count')
plt.xlabel('row_count')
plt.ylabel('Frequência')
plt.show()



In [ ]:
question_long_df = question_long_df[
    question_long_df["question_id"].isin(question_ids)
]

question_long_df.head()

In [ ]:
question_long_df.columns

In [ ]:
# Cria uma coluna combinando model_name e firac
question_long_df["model_firac"] = (
    question_long_df["model_name"] + "__" + question_long_df["firac"]
)

# Faz o pivot: linhas = question_id, colunas = model_firac, valores = is_correct
question_wide_df = question_long_df.pivot_table(
    index=["question_id", "materia"],
    columns="model_firac",
    values="is_correct",
    aggfunc="first"   # assume 1 linha por combinação; caso mais, pega a primeira
).reset_index()

# (Opcional) remove o nome do índice das colunas do pivot
question_wide_df.columns.name = None

question_wide_df.head()

In [ ]:
question_wide_df.columns

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Seleciona apenas as colunas numéricas, excluindo question_id
features = question_wide_df.drop(columns=["question_id", "materia"])

# (opcional, mas recomendado) Padronização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# DataFrame com os componentes principais
pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(X_pca.shape[1])],
    index=question_wide_df.index
)

# (opcional) adiciona question_id de volta
pca_df["question_id"] = question_wide_df["question_id"]

# Variância explicada
explained_variance = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_variance": pca.explained_variance_ratio_.cumsum()
})


explained_variance.head(20)


In [ ]:
pca_df.head()

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Merge dos dataframes
df = (
    pca_df
    .merge(question_agg_df, on="question_id", how="left")
)

df = df.dropna()

# Identifica PCs e estatísticas automaticamente
pc_cols = [c for c in df.columns if c.startswith("PC")]
stat_cols = [c for c in question_agg_df.columns if c != "question_id"]

results = []

for pc in pc_cols:
    for stat in stat_cols:
        x = df[pc]
        y = df[stat]

        pearson_corr, pearson_p = pearsonr(x, y)
        spearman_corr, spearman_p = spearmanr(x, y)

        results.append({
            "PC": pc,
            "statistic": stat,
            "pearson_r": pearson_corr,
            "pearson_p": pearson_p,
            "spearman_r": spearman_corr,
            "spearman_p": spearman_p,
        })

correlation_df = pd.DataFrame(results)

correlation_df.head(100)


In [ ]:
import pandas as pd

# Loadings: cada PC é uma combinação linear das variáveis originais
loadings_df = pd.DataFrame(
    pca.components_.T,
    index=features.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# Exibe os loadingsFILA
loadings_df


In [ ]:
# Define colunas que NÃO entram no cálculo
metadata_cols = ["question_id", "materia"]

# Seleciona apenas colunas booleanas de respostas
response_cols = question_wide_df.drop(columns=metadata_cols)

# Calcula accuracy por questão
accuracy = response_cols.eq(True).sum(axis=1) / response_cols.shape[1]


In [ ]:
import matplotlib.pyplot as plt
import itertools

# Normaliza o tamanho dos pontos (tamanho proporcional ao accuracy)
min_size = 20
max_size = 300

acc_norm = (accuracy - accuracy.min()) / (accuracy.max() - accuracy.min())
sizes = min_size + acc_norm * (max_size - min_size)

# Combinações 2D entre PC1, PC2 e PC3
pc_pairs = list(itertools.combinations(["PC1", "PC2", "PC3"], 2))

# Criação da figura com subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (pc_x, pc_y) in zip(axes, pc_pairs):
    
    
    ax.scatter(
        pca_df[pc_x],
        pca_df[pc_y],
        s=sizes,
        alpha=0.7
    )
    
    ax.set_xlabel(pc_x)
    ax.set_ylabel(pc_y)
    ax.set_title(
        f"PCA: {pc_x} vs {pc_y} "
    )

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

# Alinha matéria com PCA
materia = question_wide_df.loc[pca_df.index, "materia"]

# Cria DataFrame combinado
plot_df = pca_df[["PC1", "PC2"]].copy()
plot_df["materia"] = materia.values

# Calcula média por matéria
centroids = (
    plot_df
    .groupby("materia")[["PC1", "PC2"]]
    .mean()
    .reset_index()
)

# Plot
plt.figure(figsize=(12, 9))
plt.scatter(
    centroids["PC1"],
    centroids["PC2"],
    s=300
)

# Anota o nome da matéria em cada ponto
for _, row in centroids.iterrows():
    plt.text(
        row["PC1"],
        row["PC2"],
        row["materia"],
        fontsize=10,
        ha="center",
        va="center"
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Centroide PCA por matéria (média de PC1 e PC2)")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Alinha metadados
materia = question_wide_df.loc[pca_df.index, "materia"]

# --- cálculo do accuracy (se ainda não estiver no escopo) ---
metadata_cols = ["question_id", "materia"]
response_cols = question_wide_df.drop(columns=metadata_cols)
accuracy = response_cols.eq(True).sum(axis=1) / response_cols.shape[1]

# --- normalização do tamanho dos pontos ---
min_size = 30
max_size = 400

acc_norm = (accuracy - accuracy.min()) / (accuracy.max() - accuracy.min())
sizes = min_size + acc_norm * (max_size - min_size)

# --- plot ---
plt.figure(figsize=(14, 10))

for m in materia.unique():
    mask = materia == m
    plt.scatter(
        pca_df.loc[mask, "PC1"],
        pca_df.loc[mask, "PC2"],
        s=sizes[mask],
        alpha=0.7,
        label=m
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA: PC1 vs PC2 (cor = matéria | tamanho ∝ accuracy)")

plt.legend(
    title="Matéria",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Alinha metadados
materia = question_wide_df.loc[pca_df.index, "materia"]

# --- cálculo do accuracy ---
metadata_cols = ["question_id", "materia"]
response_cols = question_wide_df.drop(columns=metadata_cols)

accuracy = response_cols.eq(True).sum(axis=1) / response_cols.shape[1]

# --- normalização do tamanho ---
min_size = 30
max_size = 400

acc_norm = (accuracy - accuracy.min()) / (accuracy.max() - accuracy.min())
sizes = min_size + acc_norm * (max_size - min_size)

# --- limites fixos (mesma escala em todos os plots) ---
x_min, x_max = pca_df["PC1"].min(), pca_df["PC1"].max()
y_min, y_max = pca_df["PC2"].min(), pca_df["PC2"].max()

# --- loop por matéria ---
for m in materia.unique():

    plt.figure(figsize=(14, 10))

    # 🔹 outras matérias (cinza)
    other_mask = materia != m
    plt.scatter(
        pca_df.loc[other_mask, "PC1"],
        pca_df.loc[other_mask, "PC2"],
        s=sizes[other_mask],
        color="lightgray",
        alpha=0.4,
        label="Outras matérias"
    )

    # 🔹 matéria em destaque
    highlight_mask = materia == m
    plt.scatter(
        pca_df.loc[highlight_mask, "PC1"],
        pca_df.loc[highlight_mask, "PC2"],
        s=sizes[highlight_mask],
        alpha=0.9,
        label=m
    )

    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(f"PCA: destaque da matéria — {m}")

    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

# Alinha a coluna 'materia' ao PCA
materia = question_wide_df.loc[pca_df.index, "materia"]

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")

for m in materia.unique():
    mask = materia == m
    ax.scatter(
        pca_df.loc[mask, "PC1"],
        pca_df.loc[mask, "PC2"],
        pca_df.loc[mask, "PC3"],
        label=m
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("PCA: PC1 vs PC2 vs PC3 (colorido por matéria)")

# Legenda fora do gráfico
ax.legend(
    title="Matéria",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


# t-SNE

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

# Seleciona features (exclui question_id e materia)
features = question_wide_df.drop(columns=["question_id", "materia"])

# (recomendado) Padronização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# t-SNE
tsne_model = TSNE(
    n_components=2,
    perplexity=30,      # ajuste conforme o tamanho do dataset
    learning_rate=200,
    n_iter=1000,
    random_state=42,
    init="pca"
)

X_tsne = tsne_model.fit_transform(X_scaled)

# DataFrame com o embedding t-SNE
tsne_df = pd.DataFrame(
    X_tsne,
    columns=["TSNE1", "TSNE2"],
    index=question_wide_df.index
)

# adiciona metadados
tsne_df["question_id"] = question_wide_df["question_id"]
tsne_df["materia"] = question_wide_df["materia"]


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 10))

# Calcula e plota o centroide de cada matéria
for m in tsne_df["materia"].unique():
    mask = tsne_df["materia"] == m
    
    centroid_x = tsne_df.loc[mask, "TSNE1"].mean()
    centroid_y = tsne_df.loc[mask, "TSNE2"].mean()
    
    plt.scatter(
        centroid_x,
        centroid_y,
        s=200,          # tamanho maior para destacar o centroide
        alpha=0.9,
        label=m
    )
    
    # Rótulo textual do centroide
    plt.text(
        centroid_x,
        centroid_y,
        m,
        fontsize=9,
        ha="center",
        va="center"
    )

plt.xlabel("TSNE1")
plt.ylabel("TSNE2")
plt.title("t-SNE: Centroides das matérias no espaço latente")

plt.legend(
    title="Matéria",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


# UMAP

In [ ]:
import umap
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Seleciona features (exclui question_id)
features = question_wide_df.drop(columns=["question_id", "materia"])

# (recomendado) Padronização
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# UMAP
umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
)

X_umap = umap_model.fit_transform(X_scaled)

# DataFrame com o embedding UMAP
umap_df = pd.DataFrame(
    X_umap,
    columns=["UMAP1", "UMAP2"],
    index=question_wide_df.index
)

# (opcional) adiciona metadados
umap_df["question_id"] = question_wide_df["question_id"]
umap_df["materia"] = question_wide_df["materia"]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Intensidade do jitter (ajuste se necessário)
jitter_strength = 0.02

# Cria jitter
jitter_x = np.random.normal(0, jitter_strength, size=len(umap_df))
jitter_y = np.random.normal(0, jitter_strength, size=len(umap_df))

plt.figure()

for m in umap_df["materia"].unique():
    mask = umap_df["materia"] == m
    plt.scatter(
        umap_df.loc[mask, "UMAP1"] + jitter_x[mask],
        umap_df.loc[mask, "UMAP2"] + jitter_y[mask],
        label=m,
        alpha=0.7
    )

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP: Espaço latente das questões (com jitter, colorido por matéria)")

plt.legend(
    title="Matéria",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 10))

# Calcula e plota o centroide de cada matéria
for m in umap_df["materia"].unique():
    mask = umap_df["materia"] == m
    
    centroid_x = umap_df.loc[mask, "UMAP1"].mean()
    centroid_y = umap_df.loc[mask, "UMAP2"].mean()
    
    plt.scatter(
        centroid_x,
        centroid_y,
        s=200,          # tamanho maior para destacar o centroide
        alpha=0.9,
        label=m
    )
    
    # Opcional: rótulo textual do centroide
    plt.text(
        centroid_x,
        centroid_y,
        m,
        fontsize=9,
        ha="center",
        va="center"
    )

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP: Centroides das matérias no espaço latente")

plt.legend(
    title="Matéria",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()
